# Analisi dati per spettroscopia $\gamma$

A grandi linee:

1. **Realizzazione del set-up e verifica delle impostazioni**
   - Identificare le caratteristiche principali dello spettro di una sorgente ${}^{60}\text{Co}$ paragonandole ai segnali osservati con l’oscilloscopio.
   - Verificare che con la sorgente di 60Co il picco ad energia maggiore si trovi intorno al canale $780$   
2. **Studio della catena elettronica di misura (solo postazioni A e D)**
   - Verificare la linearità della catena elettronica misurando $V_{in}$ e $V_{out}$ dell'amplificatore
3. **Acquisizione degli spettri**
    - Per il ${}^{60}\text{Co}$ e per ${}^{137}\text{Cs}$ misuriamo ripetutamente per incertezza.
    - Si acquisisce uno spettro per tutti i campioni disponibili ${}^{60}\text{Co} , {}^{137}\text{Cs} , {}^{22}\text{Na}, {}^{57}\text{Co}$.
    - Acquisire il fondo ambientale
4. **Calibrazione del multicanale**
5. **Studio degli spettri delle sorgenti $\gamma$**
6. **Identificazione di sorgenti sconosciute**
   - Utilizzare la retta di calibrazione per ricavare l'energia corrispondente ai picchi fotoelettrici osservati
   - Identificare le sorgenti corrispondenti a quei picchi con la tabella fornita
7. **Studio della risoluzione dei rivelatori**
8. **Misura dell'attività di una sorgente**
   - Acquisire per un tempo "abbastanza lungo" gli spettri delle sorgenti ${}^{137}\text{Cs}$ (FS65,FS66)
   - Determinare l'attività delle sorgenti sia con il metodo relativo che quello assoluto
9. **Determinazione del coefficiente di assorbimento**
    - Acquisire il numero di conteggi netti prima senza spessori e successivamente aumentando sempre di più lo spessore di piombo
    - Determinare il coefficiente di assorbimento del piombo
10. **Analisi del picco somma nello spettro del ${}^{60}\text{Co}$**
   - Modificare l'amplificazione in modo che il picco somma cada nella scala del multicanale
   - Acquisire il numero di conteggi nel picco somma

### utils

In [ ]:
import numpy as np
from utils import fitPlotter, Zscore, texTabler, mytestZ

## Studio della catena elettronica

Questa parte dell'esperienza la fanno solo le postazioni A e D, io per sicurezza la metto. 

Le proprietà della catena elettronica di misura vengono messe in evidenza con l’uso di un generatore di impulsi, di un oscilloscopio e di un analizzatore multicanale.


**COSA DOBBIAMO MISURARE:**
- Dopo aver impostato il set-up con le indicazioni della scheda di laboratorio, misurare il canale del picco acquisito e per 8 diversi valori di $V_{in}$ misurare la corrispondente $V_{out}$.

**ANALISI DATI:**
- Verificare la linearità della catena di amplificazione con un fit lineare sui grafici di CHN in funzione di $V_{in}$ e, nella postazione A, di $V_{out}$ in funzione di $V_{in}$.




In [ ]:
# first we see CHN as a function of Vin
import numpy as np

CHN = np.array([]) 
Vin = np.array([])
err_CHN = np.array([])
err_Vin = np.array([])

CHNvsVin = fitPlotter("CHN vs Vin")
_ = CHNvsVin.addGraph(Vin, CHN, err_Vin, err_CHN, title = " CHN vs Vin plot", fit_formula = None)
CHNvsVin.drawCanvas()


In [ ]:
# here we see Vout as a function of CHN
import numpy as np

Vout = np.array([]) 
Vin = np.array([])
err_Vout = np.array([])
err_Vin = np.array([])

VoutvsVin = fitPlotter("Vout vs Vin")
_ = VoutvsVin.addGraph(Vin, Vout, err_Vin, err_Vout, title = " Vout vs Vin plot", fit_formula = None)
VoutvsVin.drawCanvas()

## Calibrazione del multicanale

Vogliamo usare le posizioni nel multicanale dei picchi fotoelettrici di tutti i materiali radioattivi per calibrare il multicanale.

In [ ]:
# we need to measure the peak's channels
# and to estimate the error

energyNa22   = 0.511
energyCo57   = 0.1221
energyCo60_1 = 1.173
energyCo60_2 = 1.333
energyCs137  = 0.6617

errenergyNa22   = 0.001
errenergyCo57   = 0.0001
errenergyCo60_1 = 0.001
errenergyCo60_2 = 0.001
errenergyCs137  = 0.0001

chnNa22   = 300
chnCo57   = 80
chnCo60_1 = 670
chnCo60_2 = 760
chnCs137  = 380

errchnNa22   = 1
errchnCo57   = 1
errchnCo60_1 = 1
errchnCo60_2 = 1
errchnCs137  = 1

energies = np.array([energyNa22, energyCo57, energyCo60_1, energyCo60_2, energyCs137])
errenergies = np.array([errenergyNa22, errenergyCo57, errenergyCo60_1, errenergyCo60_2, errenergyCs137])

channels = np.array([chnNa22, chnCo57, chnCo60_1, chnCo60_2, chnCs137])
errchannels = np.array([errchnNa22, errchnCo57, errchnCo60_1, errchnCo60_2, errchnCs137])

In [ ]:
# fit with pol2 on energy vs channel (also pol1 could work, but we're modelling nonlinear effects)

calibrationPlotter = fitPlotter("EnergyVsChannel_Calibration")
calibrationParams = calibrationPlotter.addGraph(channels, energies, errchannels, errenergies,title="Calibration; C [chn]; E [MeV]",fit_formula="pol2")
calibrationPlotter.drawCanvas(dimX=1000, dimY=500, legend=False)

In [ ]:
# checking compatibility of p0 with 0

p0    = calibrationParams[0][0]
errp0 = calibrationParams[0][1]

Z, pvalue = mytestZ(p0, errp0, 0., 0.)

print(f"p0 = {p0:.4f} +- {errp0:.4f} with 0 +- 0 ----> Z = {Z:.3f} and p = {pvalue:.3f}")

In [ ]:
# here we define the calibration function:
# we need to translate the channels into energy values!

def calibration(chns, errchns, params=calibrationParams):
    """ calibration function from channels to energies (with errors, but disregarding covariance) """
    # E = a + b * chn + c * chn^2
    a, erra = params[0][0], params[0][1]
    b, errb = params[1][0], params[1][1]
    c, errc = params[2][0], params[2][1]

    energies    = a + b * chns + c * (chns**2)
    errenergies = np.sqrt((erra**2) + (chns * errb)**2 + ((chns**2) * errc)**2 + ((b + 2*c* chns)**2) * (errchns**2))

    return energies, errenergies

# if the function works, these values should correspond to what we used to define the calibration
ens, errens = calibration(channels, errchannels)
print("energies:     ", ens)
print("energy errors:", errens)

In [ ]:
# checking value of second peak for 22-sodium

energyNa22_2    = 1.275
errenergyNa22_2 = 0.001

chnNa22_2    = 720
errchnNa22_2 = 1

measNa22_2, errmeasNa22_2 = calibration(chnNa22_2, errchnNa22_2)

Z, pvalue = mytestZ(measNa22_2, errmeasNa22_2, energyNa22_2, errenergyNa22_2)

print(f"meas = {measNa22_2:.4f} +- {errmeasNa22_2:.4f} with theo = {energyNa22_2:.4f} +- {errenergyNa22_2:.4f} --> Z = {Z:.3f} and p = {pvalue:.3f}")

## Acquisizione degli spettri

Vogliamo vedere lo spettro di emissione di alcune sorgenti, è importante acquisire ogni sorgente per un tempo sufficiente ad avere tutte le caratteristiche degli spettri da analizzare ben definite (picco fotoelettrico, spalla Compton, backscattering...).

**COSA DOBBIAMO MISURARE:**
- Acquisire una serie di almeno 8 misure della durata di 2 minuti dello spettro di ${}^{60}\text{Co}$ o di ${}^{137}\text{Cs}$ e annotare la posizione del centroide del picco fotoelettrico e la sua FWHM.
- Acquisire uno spettro per tutti i tipi di sorgenti a disposizione
- Alla fine della serie di misure, acquisire nuovamente lo spettro della sorgente utilizzata nella serie iniziale di misure (vogliamo verificare che il guadagno sia rimasto costante)
- Acquisire inoltre lo spettro del fondo ambientale

**ANALISI DATI:**
- Analizzare gli spettri acquisiti andando a studiare le caratteristiche del picco fotoelettrico, spalla Compton, backscattering e l'emissione di raggi X della transizione $K_{\alpha}$ del piombo.

In [ ]:
#code for this part

## Attività di una sorgente

Vogliamo ricavare l'attività di due sorgenti di ${}^{137}\text{Cs}$

**COSA DOBBIAMO MISURARE:**
- Acquisire per un tempo sufficientemente lungo gli spettri delle sorgenti di ${}^{137}\text{Cs}$ FS65 e FS66 posizionando le sorgenti esattamente a 9.3 cm dal rivelatore.

**ANALISI DATI:**
- Utilizzando gli spettri, ricavare l'attività di FS66 con il metodo relativo (utilizzare FS65 come sorgente nota)
- Determinare l'attività di entrambe le sorgenti con il metodo assoluto. $$ A = \frac{R}{\epsilon Gf_g} $$





In [ ]:
# code for this part

## Determinazione del coefficiente di massa

Vogliamo verificare la legge $$ I = I_0 e^{\frac{-\mu X}{\rho}} $$ e da questa ricavarne il coefficiente di massa $ \frac{\mu}{\rho}$.

**COSA DOBBIAMO MISURARE:**
- Misurare il numero di conteggi netti nel picco fotoelettrico effettuando la prima misura senza spessori e poi via via aumentando.

**ANALISI DATI:**
- Verificare la legge esponenziale e successivamente ricavare il coefficiente di massa

In [ ]:
# code for this part

## Analisi del picco somma per il ${}^{60}\text{Co}$

Per questo è meglio utilizzare la sorgente FS65 perché ha maggiore attività. Prima di procedere, cambiare l'amplificazione del segnale per far sì che il picco somma cada nella scala del multicanale.

**COSA DOBBIAMO MISURARE:**
- Acquisire il numero di conteggi nel picco e misurare l'area sottesa ai due picchi del ${}^{60}\text{Co}$ ed al picco somma

**ANALISI DATI:**
- Date le efficienze del rivelatore, la frazione di angolo solido, l'attività della sorgente e il tempo di acquisizione verificare quale delle ipotesi proposte sulla scheda di laboratorio è più corretta per le nostre misure.

In [ ]:
# calculating numbers of years that passed since last measurement of activity 

from datetime import date

d1 = date(2018, 1, 30)
d2 = date(2026, 3, 19)

diff = d2 - d1

seconds_in_year = 24 * 3600 * 365.25
time = diff.total_seconds() / seconds_in_year

In [ ]:
# calculating activity of source at present time

# the source is ...
activity_old = 19.9   # kBq
erractivity_old = 1 # kBq

halflife    =  5.2714 # years
errhalflife =  0.0001 # years

lam    = np.log(2) / halflife
errlam = np.log(2) * errhalflife / (halflife**2)

time    = time # see above
errtime = 1 / 365.25 # +- one day

activity    = activity_old * np.exp(- lam * time)
erractivity = np.sqrt((erractivity_old * np.exp(-lam*time))**2 + (activity * time * errlam)**2 + (activity * lam * errtime)**2)

print(f"activity = {activity:.5f} +- {erractivity:.5f} kBq")

In [ ]:
# turning everything in Bq and not KBq
activityBQ = 1000 * activity
erractivityBQ = 1000 * erractivity

acquisitiontime    = 2188.5 # seconds
erracquisitiontime = 0.1

DeltaT    = 10**(-6)  # temporal resolution of detector (1 us = 10^-6 s)
errDeltaT = 10**(-6)

# errors are determined by poisson (CHECK IF CORRECT)
count1      = 73915
errcount1   = np.sqrt(count1)
count2      = 81741
errcount2   = np.sqrt(count2)
countsum    = 774
errcountsum = np.sqrt(countsum)

cascade_countsum = count1 * count2 / (activityBQ * acquisitiontime)
indep_countsum   = count1 * count2 * DeltaT / acquisitiontime

errcascade_countsum = cascade_countsum * np.sqrt((1/count1) + (1/count2) + (erractivityBQ/activityBQ)**2 + (erracquisitiontime/acquisitiontime)**2)
errindep_countsum   = indep_countsum * np.sqrt((1/count1) + (1/count2) + (errDeltaT/DeltaT)**2 + (erracquisitiontime/acquisitiontime)**2)

print(f"measured    = {countsum:.1f} +- {errcountsum:.1f}")
print(f"cascade     = {cascade_countsum:.1f} +- {errcascade_countsum:.1f}")
print(f"independent = {indep_countsum:.1f} +- {errindep_countsum:.1f}")